In [22]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("RoadAccidentsAnalysis").getOrCreate()

# Load the datasets
accidents_path = "D:/ABD0008/DATA/Road_accidents.csv"
time_path = "D:/ABD0008/DATA/Road_accidents_time.csv"

df_acc = spark.read.csv(accidents_path, header=True, inferSchema=True)
df_time = spark.read.csv(time_path, header=True, inferSchema=True)

months = [
    'JANUARY', 'FEBRUARY', 'MARCH', 'APRIL', 'MAY', 'JUNE',
    'JULY', 'AUGUST', 'SEPTEMBER', 'OCTOBER', 'NOVEMBER', 'DECEMBER'
]

In [23]:
# Add column 'Total' by summing all month columns
df_acc = df_acc.withColumn("Total", F.expr(" + ".join(months)))
df_acc.select("STATE/UT", "YEAR", "Total").show(5)

+-------------+----+-----+
|     STATE/UT|YEAR|Total|
+-------------+----+-----+
|A & N Islands|2001|  181|
|A & N Islands|2002|  168|
|A & N Islands|2003|  180|
|A & N Islands|2004|  215|
|A & N Islands|2005|  206|
+-------------+----+-----+
only showing top 5 rows



In [24]:
highest_2013 = (
    df_acc.filter(F.col("YEAR") == 2013)
          .orderBy(F.desc("Total"))
          .select("STATE/UT", "YEAR", "Total")
          .first()
)

print(f"State with highest accidents in 2013: {highest_2013['STATE/UT']} ({highest_2013['Total']})")

State with highest accidents in 2013: Tamil Nadu (66238)


In [25]:
avg_monthly = df_acc.groupBy("STATE/UT").agg(
    F.round(F.mean("Total") / 12, 2).alias("Avg_Monthly_Accidents")
).orderBy(F.desc("Avg_Monthly_Accidents"))

avg_monthly.show(10)

+--------------+---------------------+
|      STATE/UT|Avg_Monthly_Accidents|
+--------------+---------------------+
|    Tamil Nadu|              5071.86|
|   Maharashtra|              3819.13|
|     Karnataka|              3480.72|
|Andhra Pradesh|              3254.89|
|        Kerala|              3084.29|
|Madhya Pradesh|              2345.28|
|     Rajasthan|              1926.51|
|       Gujarat|              1787.65|
| Uttar Pradesh|               1634.1|
|   West Bengal|              1127.23|
+--------------+---------------------+
only showing top 10 rows



In [26]:
month_wise_avg = df_acc.groupBy("STATE/UT").agg(
    *[F.round(F.mean(m), 2).alias(f"Avg_{m[:3]}") for m in months]
).orderBy("STATE/UT")

month_wise_avg.show(5)

+-----------------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+
|         STATE/UT|Avg_JAN|Avg_FEB|Avg_MAR|Avg_APR|Avg_MAY|Avg_JUN|Avg_JUL|Avg_AUG|Avg_SEP|Avg_OCT|Avg_NOV|Avg_DEC|
+-----------------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+
|    A & N Islands|  18.86|  17.57|  17.71|  18.29|  15.07|   17.0|   15.5|  16.71|   17.0|  19.07|  14.64|  19.21|
|   Andhra Pradesh| 3318.0|3189.79|3409.43| 3349.5|3674.43|3326.43|3026.71|3139.07|2965.71|3121.21|3143.79|3394.57|
|Arunachal Pradesh|  21.86|  21.29|  21.07|  18.07|  21.29|  18.79|   17.0|  17.64|  20.29|  20.71|   21.5|  22.57|
|            Assam| 379.86| 371.93| 401.64| 373.14| 365.86| 354.36| 346.07| 349.29|  345.5| 379.93| 373.29| 367.57|
|            Bihar| 505.36| 544.14| 609.86| 554.29| 713.14| 664.64| 534.79| 479.57|  452.5| 494.93|  543.0|  521.5|
+-----------------+-------+-------+-------+-------+-------+-------+-----

In [29]:
# Aggregate sum of each month
month_totals = df_acc.select([F.sum(m).alias(m) for m in months]).first().asDict()

# Find the month with the maximum sum
highest_month = sorted(month_totals, key=month_totals.get, reverse=True)[0]
print(f"Month with highest accidents: {highest_month} ({month_totals[highest_month]:,} accidents)")

Month with highest accidents: MAY (521,563 accidents)


In [31]:
import builtins

time_cols = [c for c in df_time.columns if c not in ["STATE/UT", "YEAR"]]

# 1. All-India totals (escaping column names with backticks)
all_india_time = df_time.select([F.sum(F.col(f"`{c}`")).alias(c) for c in time_cols]).first().asDict()
best_slot_all_india = builtins.max(all_india_time, key=all_india_time.get)
print(f"All-India Highest Time Slot: {best_slot_all_india} ({all_india_time[best_slot_all_india]:,} accidents)")

# 2. Karnataka totals
karnataka_time = df_time.filter(F.col("STATE/UT") == "Karnataka") \
                        .select([F.sum(F.col(f"`{c}`")).alias(c) for c in time_cols]).first().asDict()
best_slot_kar = builtins.max(karnataka_time, key=karnataka_time.get)
print(f"Karnataka Highest Time Slot: {best_slot_kar} ({karnataka_time[best_slot_kar]:,} accidents)")

All-India Highest Time Slot: 15-18 hrs (Day) (906,639 accidents)
Karnataka Highest Time Slot: 15-18 hrs (Day) (93,126 accidents)


In [32]:
state_total_accidents = df_acc.groupBy("STATE/UT").agg(
    F.sum("Total").alias("Total_Accidents_2001_2014")
).orderBy(F.desc("Total_Accidents_2001_2014"))

state_total_accidents.show(5)

+--------------+-------------------------+
|      STATE/UT|Total_Accidents_2001_2014|
+--------------+-------------------------+
|    Tamil Nadu|                   852073|
|   Maharashtra|                   641614|
|     Karnataka|                   584761|
|Andhra Pradesh|                   546821|
|        Kerala|                   518161|
+--------------+-------------------------+
only showing top 5 rows



In [33]:
# 1. State average from 2001 to 2014
state_avg = df_acc.groupBy("STATE/UT").agg(F.mean("Total").alias("Avg_Total"))

# 2. Accidents for 2014
df_2014 = df_acc.filter(F.col("YEAR") == 2014).select(F.col("STATE/UT"), F.col("Total").alias("Total_2014"))

# 3. Filter states where 2014 is below average
states_less = df_2014.join(state_avg, on="STATE/UT") \
                     .filter(F.col("Total_2014") < F.col("Avg_Total")) \
                     .select("STATE/UT", "Total_2014", F.round("Avg_Total", 2).alias("State_Avg")) \
                     .orderBy("STATE/UT")

states_less.show(20, truncate=False)

+-----------------+----------+---------+
|STATE/UT         |Total_2014|State_Avg|
+-----------------+----------+---------+
|Arunachal Pradesh|185       |242.07   |
|Chandigarh       |366       |457.93   |
|Daman & Diu      |39        |45.93    |
|Jammu & Kashmir  |5778      |5915.71  |
|Kerala           |35872     |37011.5  |
|Maharashtra      |44382     |45829.57 |
|Nagaland         |28        |55.79    |
|Puducherry       |671       |1513.5   |
|Sikkim           |130       |189.0    |
|Tripura          |716       |752.43   |
|Uttarakhand      |801       |1299.43  |
+-----------------+----------+---------+



In [36]:
import builtins

# To view year-wise peak month
for yr in range(2001, 2015):
    row = df_acc.filter(F.col("YEAR") == yr).select([F.sum(m).alias(m) for m in months]).first().asDict()
    peak = builtins.max(row, key=row.get)
    print(f"Year {yr}: {peak} ({row[peak]:,} accidents)")

Year 2001: MAY (29,978 accidents)
Year 2002: MAY (31,726 accidents)
Year 2003: MAY (30,335 accidents)
Year 2004: MAY (31,752 accidents)
Year 2005: MAY (40,126 accidents)
Year 2006: MAY (37,019 accidents)
Year 2007: MAY (39,036 accidents)
Year 2008: APRIL (38,059 accidents)
Year 2009: MAY (38,928 accidents)
Year 2010: MAY (39,821 accidents)
Year 2011: MAY (41,542 accidents)
Year 2012: MAY (38,920 accidents)
Year 2013: MAY (43,064 accidents)
Year 2014: MAY (41,404 accidents)
